In [1]:
import cadquery as cq
from cadquery import exporters
import math
from jupyter_cadquery import *

from jupyter_cadquery.replay import replay, enable_replay, disable_replay

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

versions()

Overwriting auto display for cadquery Workplane and Shape

Versions:
- jupyter_cadquery  4.0.2
- cad_viewer_widget 3.0.2
- open cascade      7.7.2.1



In [2]:
cv = open_viewer("Plate Scarf", cad_width=800, height=600)
set_default_viewer("Plate Scarf")

In [3]:
# ------------------------------------------------------------
# OPEN MORTISE-AND-TENON HALF-LAP PLATE SPLICE
#
# World coordinates:
#   X = along the plate / splice axis
#   Y = across plate width
#   Z = vertical
#
# Left plate extends toward -X
# Right plate extends toward +X
# ------------------------------------------------------------

# PLATE SECTION
plate_width = 4.0       # Y
plate_height = 6.0      # Z

left_plate_length = 72.0
right_plate_length = 72.0


# ------------------------------------------------------------
# HALF-LAP
# ------------------------------------------------------------

lap_length = 12.0
lap_depth = plate_height / 2.0


# ------------------------------------------------------------
# TENON / OPEN MORTISE
# ------------------------------------------------------------

tenon_width = 1.5      # Y
tenon_height = lap_depth
tenon_length = 4.0      # X

mortise_clearance = 0.02


# ------------------------------------------------------------
# PEG
# ------------------------------------------------------------

peg_diameter = 0.75
peg_offset = 2.0        # X from tenon shoulder

In [4]:
# ------------------------------------------------------------
# PLATE BLANKS
# ------------------------------------------------------------

# Splice is centered at X = 0.
# Each blank extends halfway into the lap zone.

left_x0 = -left_plate_length
left_x1 = lap_length / 2.0

right_x0 = -lap_length / 2.0
right_x1 = right_plate_length


# Left plate blank
left_plate_blank = cq.Solid.makeBox(
    left_x1 - left_x0,
    plate_width,
    plate_height,
    cq.Vector(
        left_x0,
        -plate_width / 2.0,
        0.0,
    ),
)


# Right plate blank
right_plate_blank = cq.Solid.makeBox(
    right_x1 - right_x0,
    plate_width,
    plate_height,
    cq.Vector(
        right_x0,
        -plate_width / 2.0,
        0.0,
    ),
)

In [5]:
# ------------------------------------------------------------
# HALF-LAP CUTS
# ------------------------------------------------------------

lap_x0 = -lap_length / 2.0
lap_x1 =  lap_length / 2.0


# Remove the TOP half of the left plate over the lap zone
left_lap_cutter = cq.Solid.makeBox(
    lap_length,
    plate_width,
    lap_depth,
    cq.Vector(
        lap_x0,
        -plate_width / 2.0,
        plate_height - lap_depth,
    ),
)

left_plate_lapped = (
    left_plate_blank
    .cut(left_lap_cutter)
    .clean()
)


# Remove the BOTTOM half of the right plate over the lap zone
right_lap_cutter = cq.Solid.makeBox(
    lap_length,
    plate_width,
    lap_depth,
    cq.Vector(
        lap_x0,
        -plate_width / 2.0,
        0.0,
    ),
)

right_plate_lapped = (
    right_plate_blank
    .cut(right_lap_cutter)
    .clean()
)

In [6]:
# ------------------------------------------------------------
# PLATE SPLICE TENON
# ------------------------------------------------------------

# Right plate retains the upper half of the half-lap.
# Extend a centered tenon leftward into the left plate.

tenon_x0 = lap_x0 - tenon_length
tenon_x1 = lap_x0

tenon_y0 = -tenon_width / 2.0
tenon_z0 = plate_height - tenon_height


plate_tenon = cq.Solid.makeBox(
    tenon_length,
    tenon_width,
    tenon_height,
    cq.Vector(
        tenon_x0,
        tenon_y0,
        tenon_z0,
    ),
)


right_plate_with_tenon = (
    right_plate_lapped
    .fuse(plate_tenon)
    .clean()
)

In [7]:
# ------------------------------------------------------------
# SECOND TENON — LEFT PLATE
# ------------------------------------------------------------

left_tenon_x0 = lap_x1
left_tenon_x1 = lap_x1 + tenon_length

left_tenon_y0 = -tenon_width / 2.0
left_tenon_z0 = 0.0


left_plate_tenon = cq.Solid.makeBox(
    tenon_length,
    tenon_width,
    tenon_height,
    cq.Vector(
        left_tenon_x0,
        left_tenon_y0,
        left_tenon_z0,
    ),
)


left_plate_with_tenon = (
    left_plate_lapped
    .fuse(left_plate_tenon)
    .clean()
)

In [8]:
# ------------------------------------------------------------
# CORRESPONDING OPEN MORTISES
# ------------------------------------------------------------

mortise_overlap = 0.05

# ------------------------------------------------------------
# LEFT PLATE MORTISE
# Receives the upper tenon from the right plate.
# Open through the TOP face.
# ------------------------------------------------------------

left_mortise_x0 = tenon_x0 - mortise_clearance / 2.0
left_mortise_x1 = tenon_x1 + mortise_overlap

left_mortise_y0 = (
    -tenon_width / 2.0
    - mortise_clearance / 2.0
)

left_mortise_z0 = (
    plate_height
    - tenon_height
    - mortise_clearance / 2.0
)

left_open_mortise = cq.Solid.makeBox(
    left_mortise_x1 - left_mortise_x0,
    tenon_width + mortise_clearance,
    tenon_height + mortise_clearance,
    cq.Vector(
        left_mortise_x0,
        left_mortise_y0,
        left_mortise_z0,
    ),
)


# ------------------------------------------------------------
# RIGHT PLATE MORTISE
# Receives the lower tenon from the left plate.
# Open through the BOTTOM face.
# ------------------------------------------------------------

right_mortise_x0 = left_tenon_x0 - mortise_overlap
right_mortise_x1 = (
    left_tenon_x1
    + mortise_clearance / 2.0
)

right_mortise_y0 = (
    -tenon_width / 2.0
    - mortise_clearance / 2.0
)

right_mortise_z0 = -mortise_clearance / 2.0

right_open_mortise = cq.Solid.makeBox(
    right_mortise_x1 - right_mortise_x0,
    tenon_width + mortise_clearance,
    tenon_height + mortise_clearance,
    cq.Vector(
        right_mortise_x0,
        right_mortise_y0,
        right_mortise_z0,
    ),
)

In [9]:
# ------------------------------------------------------------
# FINISHED SPLICE MEMBERS
# ------------------------------------------------------------

left_plate_splice = (
    left_plate_with_tenon
    .cut(left_open_mortise)
    .clean()
)

right_plate_splice = (
    right_plate_with_tenon
    .cut(right_open_mortise)
    .clean()
)

In [10]:
# ------------------------------------------------------------
# PEG BORES
# ------------------------------------------------------------

peg_radius = peg_diameter / 2.0
bore_margin = 1.0

# Upper tenon:
# right plate tenon projects left from lap_x0
upper_peg_x = lap_x0 - peg_offset
upper_peg_z = plate_height - tenon_height / 2.0

# Lower tenon:
# left plate tenon projects right from lap_x1
lower_peg_x = lap_x1 + peg_offset
lower_peg_z = tenon_height / 2.0


# ------------------------------------------------------------
# BORE CUTTERS
# Axis = Y
# ------------------------------------------------------------

upper_peg_bore = cq.Solid.makeCylinder(
    peg_radius,
    plate_width + 2.0 * bore_margin,
    cq.Vector(
        upper_peg_x,
        -plate_width / 2.0 - bore_margin,
        upper_peg_z,
    ),
    cq.Vector(0, 1, 0),
)

lower_peg_bore = cq.Solid.makeCylinder(
    peg_radius,
    plate_width + 2.0 * bore_margin,
    cq.Vector(
        lower_peg_x,
        -plate_width / 2.0 - bore_margin,
        lower_peg_z,
    ),
    cq.Vector(0, 1, 0),
)

In [11]:
# ------------------------------------------------------------
# PEGGED SPLICE MEMBERS
# ------------------------------------------------------------

left_plate_final = (
    left_plate_splice
    .cut(upper_peg_bore)
    .cut(lower_peg_bore)
    .clean()
)

right_plate_final = (
    right_plate_splice
    .cut(upper_peg_bore)
    .cut(lower_peg_bore)
    .clean()
)

In [12]:
show_object(right_plate_final, name="right_plate")
show_object(left_plate_final, name="left_plate")

+
++
